In [ ]:
!pip install -U streamlit pyngrok pymongo bcrypt send-mail

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 57.8 MB/s eta 0:00:00


In [ ]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini:\n")

if not os.environ.get("Ngrok_API_KEY"):
    os.environ["Ngrok_API_KEY"] = getpass.getpass("Enter API key for ngrok:\n")

Enter API key for Google Gemini:
··········
Enter API key for ngrok:
··········


In [ ]:
mongodb_uri = getpass.getpass("Enter mongodb uri: ")
!echo "MONGODB_URI={mongodb_uri}" > .env

email_address = getpass.getpass("Enter sender email id: ")
!echo "EMAIL_ADDRESS={email_address}" >> .env

email_password = getpass.getpass("Enter sender email password: ")
!echo "EMAIL_PASSWORD={email_password}" >> .env

Enter mongodb uri: ··········
Enter sender email id: ··········
Enter sender email password: ··········


In [ ]:
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
import os
from dotenv import load_dotenv
load_dotenv(".env")
import warnings
warnings.filterwarnings("ignore")

# Create a new client and connect to the server
client = MongoClient(os.environ.get("MONGODB_URI"), server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [ ]:
%%writefile app.py
"""
DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR - Main Application
"""

import streamlit as st
import bcrypt
import json
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from datetime import datetime, date, timedelta
from zoneinfo import ZoneInfo
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from pydantic import BaseModel
from typing import List, Optional
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# ─────────────────────────────────────────────
# CONFIG & INIT
# ─────────────────────────────────────────────
load_dotenv(".env")

st.set_page_config(
    page_title="DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR",
    page_icon="🥗",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─────────────────────────────────────────────
# CUSTOM CSS
# ─────────────────────────────────────────────
st.markdown("""
<style>
/* ---------- Base & Font ---------- */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

html, body, [class*="css"] {
    font-family: 'Inter', sans-serif;
}

/* ---------- Main background ---------- */
.stApp {
    background: linear-gradient(135deg, #0f0f1a 0%, #1a1a2e 50%, #16213e 100%);
    min-height: 100vh;
}

/* ---------- Sidebar ---------- */
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #0d1117 0%, #161b22 100%) !important;
    border-right: 1px solid rgba(99, 179, 237, 0.15);
}
[data-testid="stSidebar"] .stRadio label {
    color: #e2e8f0 !important;
    font-weight: 500;
}

/* ---------- Cards ---------- */
.glass-card {
    background: rgba(255,255,255,0.05);
    backdrop-filter: blur(16px);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 16px;
    padding: 24px;
    margin: 12px 0;
    box-shadow: 0 8px 32px rgba(0,0,0,0.3);
}

.stat-card {
    background: linear-gradient(135deg, rgba(99,179,237,0.15), rgba(154,117,234,0.15));
    border: 1px solid rgba(99,179,237,0.3);
    border-radius: 12px;
    padding: 20px;
    text-align: center;
}

.metric-value {
    font-size: 2.2rem;
    font-weight: 700;
    background: linear-gradient(135deg, #63b3ed, #9a75ea);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.metric-label {
    font-size: 0.85rem;
    color: #a0aec0;
    margin-top: 4px;
    text-transform: uppercase;
    letter-spacing: 0.05em;
}

/* ---------- Section header ---------- */
.section-header {
    background: linear-gradient(135deg, #63b3ed, #9a75ea);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 1.8rem;
    font-weight: 700;
    margin-bottom: 8px;
}

/* ---------- Buttons ---------- */
.stButton > button {
    background: linear-gradient(135deg, #4f8ef7, #9a75ea) !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
    padding: 10px 24px !important;
    font-weight: 600 !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 4px 15px rgba(79,142,247,0.3) !important;
}
.stButton > button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 25px rgba(79,142,247,0.5) !important;
}

/* ---------- Inputs — white background, dark text ---------- */
.stTextInput > div > input,
.stNumberInput > div > input,
.stTextArea > div > textarea {
    background: #ffffff !important;
    border: 1px solid rgba(255,255,255,0.15) !important;
    border-radius: 8px !important;
    color: #1a1a2e !important;
}
.stTextInput > div > input::placeholder,
.stNumberInput > div > input::placeholder,
.stTextArea > div > textarea::placeholder {
    color: #9ca3af !important;
}

/* Selectbox / dropdown — white background, dark text */
.stSelectbox > div > div,
[data-baseweb="select"] > div {
    background: #ffffff !important;
    border-radius: 8px !important;
}
[data-baseweb="select"] span,
[data-baseweb="select"] div,
[data-baseweb="select"] input {
    color: #1a1a2e !important;
    background: #ffffff !important;
}

/* Dropdown menu options list — force black text on white */
[data-baseweb="popover"],
[data-baseweb="popover"] *,
[data-baseweb="menu"],
[data-baseweb="menu"] *,
[data-baseweb="list"],
[data-baseweb="list"] *,
ul[role="listbox"],
ul[role="listbox"] *,
li[role="option"],
li[role="option"] *,
[role="option"],
[role="option"] * {
    color: #1a1a2e !important;
    background-color: #ffffff !important;
}
[role="option"]:hover,
[role="option"][aria-selected="true"],
[data-baseweb="menu"] li:hover {
    background-color: #dbeafe !important;
    color: #1e3a8a !important;
}

/* Extra nuclear override for Streamlit dropdown portal text */
div[data-testid="stSelectbox"] ~ div *,
[data-baseweb="tooltip"] *,
[class*="menu"] *,
[class*="option"] *,
[class*="MenuList"] *,
[class*="singleValue"],
[class*="placeholder"] {
    color: #1a1a2e !important;
}
/* Ensure the entire popover/portal has white bg */
[data-testid="stSelectboxVirtualDropdown"] * {
    background: #ffffff !important;
    color: #1a1a2e !important;
}
/* Streamlit uses a global portal div for dropdowns */
body > div[class]:not([data-testid]) ul,
body > div[class]:not([data-testid]) li,
body > div[class]:not([data-testid]) span {
    color: #1a1a2e !important;
    background-color: #ffffff !important;
}

/* Date & time inputs */
.stDateInput > div > div > input,
.stTimeInput > div > div > input {
    background: #ffffff !important;
    color: #1a1a2e !important;
    border-radius: 8px !important;
}
.stDateInput > div > div > input::placeholder,
.stTimeInput > div > div > input::placeholder {
    color: #9ca3af !important;
}

/* Multiselect */
.stMultiSelect [data-baseweb="select"] > div {
    background: #ffffff !important;
}
.stMultiSelect span, .stMultiSelect div {
    color: #1a1a2e !important;
}
.stMultiSelect [data-baseweb="tag"] {
    background: rgba(79,142,247,0.15) !important;
}
.stMultiSelect [data-baseweb="tag"] span {
    color: #4f8ef7 !important;
}

/* ---------- Tabs ---------- */
.stTabs [data-baseweb="tab"] {
    color: #a0aec0;
    font-weight: 500;
}
.stTabs [aria-selected="true"] {
    color: #63b3ed !important;
    border-bottom-color: #63b3ed !important;
}

/* ---------- Logo area ---------- */
.app-logo {
    text-align: center;
    padding: 16px 0 8px;
}
.app-logo h2 {
    background: linear-gradient(135deg, #63b3ed, #9a75ea, #f093fb);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 1.1rem;
    font-weight: 700;
    line-height: 1.3;
}

/* ---------- Alerts / Success ---------- */
.success-banner {
    background: rgba(72, 199, 142, 0.15);
    border: 1px solid rgba(72, 199, 142, 0.4);
    border-radius: 10px;
    padding: 14px 20px;
    color: #48c78e;
    font-weight: 500;
}
.warning-banner {
    background: rgba(255, 152, 0, 0.15);
    border: 1px solid rgba(255, 152, 0, 0.4);
    border-radius: 10px;
    padding: 14px 20px;
    color: #ff9800;
    font-weight: 500;
}
.error-banner {
    background: rgba(239, 68, 68, 0.15);
    border: 1px solid rgba(239, 68, 68, 0.4);
    border-radius: 10px;
    padding: 14px 20px;
    color: #ef4444;
    font-weight: 500;
}

/* ---------- Progress bar ---------- */
.stProgress > div > div {
    background: linear-gradient(90deg, #63b3ed, #9a75ea) !important;
}

/* ---------- Expander ---------- */
.streamlit-expanderHeader {
    background: rgba(255,255,255,0.05) !important;
    border-radius: 8px !important;
    color: #e2e8f0 !important;
}

/* ---------- Slider ---------- */
.stSlider > div > div > div {
    background: linear-gradient(90deg, #63b3ed, #9a75ea) !important;
}

/* ══════════════════════════════════════════
   GLOBAL WHITE TEXT OVERRIDES
   ══════════════════════════════════════════ */

/* All base text */
p, span, div, li, td, th, label, caption {
    color: #e2e8f0;
}

/* Headings */
h1, h2, h3, h4, h5, h6 {
    color: #f0f4ff !important;
}

/* Markdown rendered text */
.stMarkdown p, .stMarkdown li, .stMarkdown span {
    color: #e2e8f0 !important;
}
.stMarkdown h1, .stMarkdown h2, .stMarkdown h3, .stMarkdown h4 {
    color: #f0f4ff !important;
}
.stMarkdown strong {
    color: #ffffff !important;
}
.stMarkdown em {
    color: #cbd5e0 !important;
}
.stMarkdown a {
    color: #63b3ed !important;
}

/* Table in markdown */
.stMarkdown table { width: 100%; border-collapse: collapse; }
.stMarkdown th {
    color: #ffffff !important;
    background: rgba(99,179,237,0.15) !important;
    padding: 10px 14px !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
}
.stMarkdown td {
    color: #e2e8f0 !important;
    padding: 8px 14px !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
}
.stMarkdown tr:nth-child(even) td { background: rgba(255,255,255,0.03) !important; }
.stMarkdown blockquote {
    border-left: 3px solid #63b3ed !important;
    padding-left: 16px !important;
    color: #cbd5e0 !important;
    background: rgba(99,179,237,0.07) !important;
    border-radius: 0 8px 8px 0 !important;
}

/* Form labels */
.stTextInput label, .stNumberInput label, .stSelectbox label,
.stTextArea label, .stFileUploader label, .stDateInput label,
.stTimeInput label, .stSlider label, .stRadio label,
.stCheckbox label, .stMultiSelect label {
    color: #e2e8f0 !important;
    font-weight: 500 !important;
}

/* Selectbox text — handled above in the inputs block */

/* Radio buttons */
.stRadio div[role="radiogroup"] label {
    color: #e2e8f0 !important;
}

/* Slider value */
.stSlider [data-testid="stTickBar"] span,
.stSlider div[data-testid="stSlider"] p {
    color: #e2e8f0 !important;
}

/* Metric widget */
[data-testid="stMetricValue"] {
    color: #ffffff !important;
    font-weight: 700 !important;
}
[data-testid="stMetricLabel"] {
    color: #a0aec0 !important;
}
[data-testid="stMetricDelta"] {
    color: #48c78e !important;
}

/* Info / Warning / Error / Success boxes */
[data-testid="stAlert"] p,
[data-testid="stAlert"] div {
    color: #ffffff !important;
}
.stAlert p { color: #ffffff !important; }

/* Dataframe */
[data-testid="stDataFrame"] { color: #e2e8f0 !important; }

/* Expander */
details summary p,
[data-testid="stExpander"] summary p,
[data-testid="stExpander"] summary span {
    color: #e2e8f0 !important;
    font-weight: 500 !important;
}
[data-testid="stExpander"] p,
[data-testid="stExpander"] li {
    color: #e2e8f0 !important;
}

/* Tabs */
.stTabs [data-baseweb="tab"] p,
.stTabs [data-baseweb="tab"] span {
    color: #a0aec0 !important;
}
.stTabs [aria-selected="true"] p,
.stTabs [aria-selected="true"] span {
    color: #63b3ed !important;
}

/* Divider */
hr { border-color: rgba(255,255,255,0.1) !important; }

/* Sidebar text */
[data-testid="stSidebar"] p,
[data-testid="stSidebar"] span,
[data-testid="stSidebar"] label {
    color: #e2e8f0 !important;
}
[data-testid="stSidebar"] h1,
[data-testid="stSidebar"] h2,
[data-testid="stSidebar"] h3 {
    color: #f0f4ff !important;
}

/* Progress label */
[data-testid="stProgress"] p { color: #e2e8f0 !important; }

/* Caption */
.stCaption, [data-testid="stCaptionContainer"] p { color: #a0aec0 !important; }

/* Column headers in dataframe */
[data-testid="stDataFrame"] th { color: #ffffff !important; background: rgba(99,179,237,0.12) !important; }
[data-testid="stDataFrame"] td { color: #e2e8f0 !important; }
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# TIMEZONE HELPER
# ─────────────────────────────────────────────
def now_ist():
    return datetime.now(ZoneInfo("Asia/Kolkata"))

# ─────────────────────────────────────────────
# MONGODB CONNECTION
# ─────────────────────────────────────────────
@st.cache_resource
def get_db():
    client = MongoClient(os.environ.get("MONGODB_URI"), server_api=ServerApi('1'))
    db = client["CalorieDB"]
    return db

db = get_db()
users_col     = db["users"]
profiles_col  = db["user_profiles"]
meals_col     = db["meals"]

# ─────────────────────────────────────────────
# GEMINI CLIENT
# ─────────────────────────────────────────────
@st.cache_resource
def get_gemini():
    return genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

gemini_client = get_gemini()

# ─────────────────────────────────────────────
# PYDANTIC SCHEMAS
# ─────────────────────────────────────────────
class NutrientPerUnit(BaseModel):
    unit: str
    calories_kcal: float
    protein_g: float
    carbs_g: float
    fats_g: float
    fiber_g: float

class FoodItem(BaseModel):
    food_name: str
    default_quantity: float
    nutrients_per_unit: NutrientPerUnit

class FoodDetectionResponse(BaseModel):
    foods: List[FoodItem]

class AIInsights(BaseModel):
    summary: str
    health_score: float
    suggestions: List[str]
    positives: List[str]
    warnings: List[str]

# ─────────────────────────────────────────────
# SESSION STATE INIT
# ─────────────────────────────────────────────
def init_session_state():
    defaults = {
        "authenticated": False,
        "username": None,
        "user_email": None,
        "user_id": None,
        "profile_complete": False,
        "auth_mode": "login",
        # tracker states
        "food_detection_result": None,
        "quantities_dict": {},
        "tracker_totals": None,
        "tracker_insights": None,
        "tracker_submitted": False,
        "current_section": "🏠 Home",
    }
    for k, v in defaults.items():
        if k not in st.session_state:
            st.session_state[k] = v

init_session_state()

# ─────────────────────────────────────────────
# AUTH HELPERS
# ─────────────────────────────────────────────
def hash_pw(pw: str) -> str:
    return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()

def verify_pw(pw: str, hashed: str) -> bool:
    return bcrypt.checkpw(pw.encode(), hashed.encode())

def login_user(email: str, password: str):
    user = users_col.find_one({"email": email.lower().strip()})
    if not user:
        return False, "No account found with this email."
    if not verify_pw(password, user["password"]):
        return False, "Incorrect password."
    # Check if profile complete
    profile = profiles_col.find_one({"user_id": str(user["_id"])})
    st.session_state.authenticated = True
    st.session_state.username = user["name"]
    st.session_state.user_email = user["email"]
    st.session_state.user_id = str(user["_id"])
    st.session_state.profile_complete = profile is not None and profile.get("daily_calorie_target") is not None
    return True, "Login successful!"

def signup_user(name: str, email: str, password: str):
    if users_col.find_one({"email": email.lower().strip()}):
        return False, "Email already registered."
    doc = {
        "name": name.strip(),
        "email": email.lower().strip(),
        "password": hash_pw(password),
        "created_at": now_ist().isoformat(),
    }
    result = users_col.insert_one(doc)
    return True, str(result.inserted_id)

# ─────────────────────────────────────────────
# BACKEND CALCULATION
# ─────────────────────────────────────────────
def calculate_totals(food_items: List[FoodItem], quantities_dict: dict) -> dict:
    total = {"calories_kcal": 0, "protein_g": 0, "carbs_g": 0, "fats_g": 0, "fiber_g": 0}
    for food in food_items:
        qty = quantities_dict.get(food.food_name, food.default_quantity)
        p = food.nutrients_per_unit
        total["calories_kcal"] += p.calories_kcal * qty
        total["protein_g"]     += p.protein_g     * qty
        total["carbs_g"]       += p.carbs_g       * qty
        total["fats_g"]        += p.fats_g        * qty
        total["fiber_g"]       += p.fiber_g       * qty
    return {k: round(v, 2) for k, v in total.items()}

# ─────────────────────────────────────────────
# GEMINI HELPERS
# ─────────────────────────────────────────────
def detect_foods_from_text(user_text: str) -> FoodDetectionResponse:
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=user_text,
        config={
            "system_instruction": (
                "You are a certified nutritionist. Identify all food items mentioned. "
                "Return realistic per-unit nutrients. Use standard units: piece, cup, gram, bowl, plate, slice. "
                "Be accurate with calorie estimates based on average Indian/global portion sizes."
            ),
            "response_mime_type": "application/json",
            "response_schema": FoodDetectionResponse,
        },
    )
    return response.parsed

def detect_foods_from_image(image_bytes: bytes, mime_type: str = "image/jpeg") -> FoodDetectionResponse:
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[
            types.Part.from_bytes(data=image_bytes, mime_type=mime_type),
            "Identify all food items visible in the image. Estimate portion sizes. Return per-unit nutrients.",
        ],
        config={
            "system_instruction": (
                "You are a certified nutritionist. Identify food items from the image. "
                "Estimate realistic portion sizes and return per-unit nutrients. "
                "Use standard units: piece, cup, gram, bowl, plate, slice."
            ),
            "response_mime_type": "application/json",
            "response_schema": FoodDetectionResponse,
        },
    )
    return response.parsed

def get_ai_insights(totals: dict, profile: dict, meal_type: str, food_items: list = None) -> AIInsights:
    foods_list = ""
    if food_items:
        for f in food_items:
            foods_list += f"\n- {f.food_name} (calories: {f.nutrients_per_unit.calories_kcal:.0f} kcal/{f.nutrients_per_unit.unit}, protein: {f.nutrients_per_unit.protein_g:.1f}g, carbs: {f.nutrients_per_unit.carbs_g:.1f}g, fats: {f.nutrients_per_unit.fats_g:.1f}g)"

    prompt = f"""
You are a certified nutritionist and health coach. Analyse this meal for the user based on their health profile.

USER PROFILE:
- Age: {profile.get('age')} years | Gender: {profile.get('gender')}
- Weight: {profile.get('weight_kg')} kg | Height: {profile.get('height_cm')} cm | BMI: {profile.get('bmi')}
- Activity Level: {profile.get('activity_level')}
- Health Goal: {profile.get('goal')}
- Daily Calorie Target: {profile.get('daily_calorie_target')} kcal
- Dietary Preference: {profile.get('diet_type')}
- Medical Conditions: {profile.get('medical_conditions', 'None')}
- Allergies: {profile.get('allergies', 'None')}

MEAL TYPE: {meal_type}

FOOD ITEMS IN THIS MEAL:{foods_list if foods_list else " (see totals below)"}

TOTAL NUTRITION:
- Calories: {totals['calories_kcal']} kcal (target: {profile.get('daily_calorie_target')} kcal/day)
- Protein: {totals['protein_g']} g
- Carbohydrates: {totals['carbs_g']} g
- Fats: {totals['fats_g']} g
- Fiber: {totals['fiber_g']} g

INSTRUCTIONS:
1. summary: Write a 2-3 sentence analysis of this specific meal — what it contains, its overall nutritional character, and whether it fits their goal of "{profile.get('goal')}".
2. health_score: Rate this meal 0–10 considering: calorie appropriateness for their goal, macro balance, food quality, and alignment with their dietary preference and medical conditions.
3. positives: List 3–5 specific health benefits of the actual food items eaten (e.g., "Dal is rich in plant protein supporting muscle repair", "Banana provides quick energy for post-workout recovery").
4. suggestions: List 3–5 specific actionable suggestions — what to add, reduce, or swap to better align with their goal of "{profile.get('goal')}" (be food-specific, e.g., "Add a handful of spinach to boost iron intake").
5. warnings: List any concerns — foods that may conflict with their medical conditions, allergies, dietary preference, or goal. If none, note one general moderation tip instead.

Be specific to the actual foods eaten. Do NOT give generic nutrition advice.
"""
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": AIInsights,
        },
    )
    return response.parsed

def get_profile_insights(profile: dict) -> str:
    prompt = f"""
Based on the following user health profile, provide comprehensive insights including:
1. Recommended daily calorie intake (be very specific with a number)
2. Macro nutrient targets (protein, carbs, fats in grams)
3. Personalized health tips (at least 5)
4. Foods to include and avoid
5. Hydration recommendations
6. Exercise suggestions aligned with their goal
7. Weekly weight loss/gain projection if they follow recommendations

User Profile:
- Name: {profile.get('name')}
- Age: {profile.get('age')} years
- Gender: {profile.get('gender')}
- Weight: {profile.get('weight_kg')} kg
- Height: {profile.get('height_cm')} cm
- Activity Level: {profile.get('activity_level')}
- Health Goal: {profile.get('goal')}
- Dietary Preference: {profile.get('diet_type')}
- Medical Conditions: {profile.get('medical_conditions', 'None')}
- Allergies: {profile.get('allergies', 'None')}
- Sleep Hours: {profile.get('sleep_hours', 7)} hours/day

Format the response in clear Markdown with sections, bullet points, and emojis for readability.
Be specific, personalized, and encouraging.
"""
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt,
    )
    return response.text

# ─────────────────────────────────────────────
# EMAIL HELPER
# ─────────────────────────────────────────────
def send_calorie_alert_email(user_email: str, username: str, consumed: float, target: float, meal_insights: AIInsights):
    try:
        from send_mail import SendMail
        overage = consumed - target
        html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<style>
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #0f0f1a; color: #e2e8f0; margin: 0; padding: 0; }}
  .wrapper {{ max-width: 600px; margin: 0 auto; padding: 20px; }}
  .header {{ background: linear-gradient(135deg, #4f8ef7, #9a75ea); border-radius: 16px 16px 0 0; padding: 32px; text-align: center; }}
  .header h1 {{ color: white; margin: 0; font-size: 24px; }}
  .header p {{ color: rgba(255,255,255,0.85); margin: 8px 0 0; }}
  .body {{ background: #161b22; border: 1px solid rgba(255,255,255,0.1); border-radius: 0 0 16px 16px; padding: 32px; }}
  .alert-box {{ background: rgba(239,68,68,0.15); border: 1px solid rgba(239,68,68,0.4); border-radius: 10px; padding: 20px; margin-bottom: 24px; text-align: center; }}
  .alert-box h2 {{ color: #ef4444; margin: 0 0 8px; font-size: 20px; }}
  .stats-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 24px; }}
  .stat {{ background: rgba(255,255,255,0.05); border-radius: 10px; padding: 16px; text-align: center; }}
  .stat .val {{ font-size: 28px; font-weight: 700; color: #63b3ed; }}
  .stat .lbl {{ font-size: 12px; color: #a0aec0; margin-top: 4px; text-transform: uppercase; }}
  .section {{ margin-bottom: 20px; }}
  .section h3 {{ color: #9a75ea; margin: 0 0 12px; font-size: 16px; }}
  ul {{ margin: 0; padding-left: 20px; }}
  li {{ margin-bottom: 8px; color: #cbd5e0; font-size: 14px; }}
  .footer {{ text-align: center; margin-top: 24px; color: #718096; font-size: 12px; }}
  .health-score {{ font-size: 48px; font-weight: 800; color: {'#48c78e' if meal_insights.health_score >= 7 else '#f6ad55' if meal_insights.health_score >= 5 else '#ef4444'}; }}
</style>
</head>
<body>
<div class="wrapper">
  <div class="header">
    <h1>🚨 Daily Calorie Alert</h1>
    <p>Hey {username}, you've exceeded your daily calorie target!</p>
  </div>
  <div class="body">
    <div class="alert-box">
      <h2>⚠️ Calorie Limit Exceeded</h2>
      <p style="color:#a0aec0; margin:0;">You are <strong style="color:#ef4444;">{overage:.0f} kcal</strong> over your daily target.</p>
    </div>

    <div class="stats-grid">
      <div class="stat"><div class="val">{consumed:.0f}</div><div class="lbl">Consumed (kcal)</div></div>
      <div class="stat"><div class="val">{target:.0f}</div><div class="lbl">Target (kcal)</div></div>
    </div>

    <div class="section" style="text-align:center; margin-bottom:24px;">
      <p style="color:#a0aec0; font-size:14px; margin-bottom:8px;">Latest Meal Health Score</p>
      <div class="health-score">{meal_insights.health_score:.1f}<span style="font-size:24px;color:#a0aec0;">/10</span></div>
    </div>

    <div class="section">
      <h3>📝 Meal Summary</h3>
      <p style="color:#cbd5e0; font-size:14px;">{meal_insights.summary}</p>
    </div>

    <div class="section">
      <h3>✅ What You Did Well</h3>
      <ul>{''.join(f'<li>{p}</li>' for p in meal_insights.positives)}</ul>
    </div>

    <div class="section">
      <h3>💡 Suggestions for Rest of Day</h3>
      <ul>{''.join(f'<li>{s}</li>' for s in meal_insights.suggestions)}</ul>
    </div>

    <div class="section">
      <h3>⚠️ Warnings</h3>
      <ul>{''.join(f'<li>{w}</li>' for w in meal_insights.warnings)}</ul>
    </div>

    <div class="footer">
      <p>🥗 DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR &bull; Stay healthy, stay consistent!</p>
      <p>This email was sent on {now_ist().strftime('%d %b %Y at %I:%M %p IST')}</p>
    </div>
  </div>
</div>
</body>
</html>
"""
        mail = SendMail(
            user_email,
            f"🚨 Calorie Alert: You've exceeded your daily target by {overage:.0f} kcal",
            f"Hey {username}, you have consumed {consumed:.0f} kcal which exceeds your daily target of {target:.0f} kcal.",
        )
        mail.add_html_message(html)
        mail.send()
        return True
    except Exception as e:
        st.warning(f"Email notification failed: {e}")
        return False

def send_daily_summary_email(user_email: str, username: str, profile: dict, meals_today: list, total_today: dict):
    try:
        from send_mail import SendMail
        target = profile.get("daily_calorie_target", 2000)
        consumed = total_today.get("calories_kcal", 0)
        remaining = target - consumed
        status_color = "#48c78e" if consumed <= target else "#ef4444"
        status_text = "On Track! 🎉" if consumed <= target else f"Exceeded by {consumed-target:.0f} kcal ⚠️"

        meals_html = ""
        for m in meals_today:
            meals_html += f"""
            <tr>
              <td style="padding:10px; color:#e2e8f0;">{m.get('meal_type','N/A')}</td>
              <td style="padding:10px; color:#63b3ed; text-align:center;">{m.get('final_totals',{}).get('calories_kcal',0):.0f}</td>
              <td style="padding:10px; color:#9a75ea; text-align:center;">{m.get('final_totals',{}).get('protein_g',0):.1f}g</td>
              <td style="padding:10px; color:#48c78e; text-align:center;">{m.get('final_totals',{}).get('carbs_g',0):.1f}g</td>
              <td style="padding:10px; color:#f6ad55; text-align:center;">{m.get('final_totals',{}).get('fats_g',0):.1f}g</td>
            </tr>"""

        html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<style>
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #0f0f1a; color: #e2e8f0; margin:0; padding:0; }}
  .wrapper {{ max-width: 640px; margin: 0 auto; padding: 20px; }}
  .header {{ background: linear-gradient(135deg, #4f8ef7, #9a75ea); border-radius: 16px 16px 0 0; padding: 32px; text-align: center; }}
  .header h1 {{ color: white; margin: 0; font-size: 26px; }}
  .header p {{ color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px; }}
  .body {{ background: #161b22; border: 1px solid rgba(255,255,255,0.1); border-radius: 0 0 16px 16px; padding: 32px; }}
  .status-box {{ background: rgba(255,255,255,0.05); border: 1px solid {status_color}44; border-radius: 12px; padding: 20px; text-align: center; margin-bottom: 24px; }}
  .stats-grid {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; margin-bottom: 24px; }}
  .stat {{ background: rgba(255,255,255,0.05); border-radius: 10px; padding: 14px; text-align: center; }}
  .stat .val {{ font-size: 22px; font-weight: 700; }}
  .stat .lbl {{ font-size: 11px; color: #a0aec0; margin-top: 4px; text-transform: uppercase; }}
  table {{ width: 100%; border-collapse: collapse; margin-bottom: 24px; }}
  th {{ background: rgba(255,255,255,0.08); color: #a0aec0; padding: 10px; font-size: 12px; text-transform: uppercase; letter-spacing: 0.05em; }}
  tr:nth-child(even) {{ background: rgba(255,255,255,0.03); }}
  .footer {{ text-align: center; margin-top: 24px; color: #718096; font-size: 12px; }}
</style>
</head>
<body>
<div class="wrapper">
  <div class="header">
    <h1>📊 Daily Nutrition Summary</h1>
    <p>Hello {username}! Here's how your day went — {now_ist().strftime('%d %B %Y')}</p>
  </div>
  <div class="body">
    <div class="status-box">
      <p style="color:#a0aec0; margin:0 0 8px; font-size:14px;">Today's Status</p>
      <p style="color:{status_color}; font-size:22px; font-weight:700; margin:0;">{status_text}</p>
    </div>

    <div class="stats-grid">
      <div class="stat"><div class="val" style="color:#63b3ed;">{consumed:.0f}</div><div class="lbl">Consumed</div></div>
      <div class="stat"><div class="val" style="color:#9a75ea;">{target:.0f}</div><div class="lbl">Target</div></div>
      <div class="stat"><div class="val" style="color:{'#48c78e' if remaining >= 0 else '#ef4444'};">{abs(remaining):.0f}</div><div class="lbl">{'Remaining' if remaining >= 0 else 'Exceeded'}</div></div>
    </div>

    <h3 style="color:#9a75ea; margin-bottom:12px;">🍽️ Macro Breakdown</h3>
    <div class="stats-grid" style="grid-template-columns:repeat(3,1fr); margin-bottom:24px;">
      <div class="stat"><div class="val" style="color:#9a75ea;">{total_today.get('protein_g',0):.1f}g</div><div class="lbl">Protein</div></div>
      <div class="stat"><div class="val" style="color:#63b3ed;">{total_today.get('carbs_g',0):.1f}g</div><div class="lbl">Carbs</div></div>
      <div class="stat"><div class="val" style="color:#f6ad55;">{total_today.get('fats_g',0):.1f}g</div><div class="lbl">Fats</div></div>
    </div>

    <h3 style="color:#9a75ea; margin-bottom:12px;">🥘 Today's Meals ({len(meals_today)} logged)</h3>
    <table>
      <tr><th>Meal</th><th>Calories</th><th>Protein</th><th>Carbs</th><th>Fats</th></tr>
      {meals_html if meals_html else '<tr><td colspan="5" style="text-align:center; color:#718096; padding:20px;">No meals logged today</td></tr>'}
    </table>

    <div class="footer">
      <p>🥗 DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR &bull; Keep up the great work!</p>
      <p>Sent on {now_ist().strftime('%d %b %Y at %I:%M %p IST')}</p>
    </div>
  </div>
</div>
</body>
</html>
"""
        mail = SendMail(
            user_email,
            f"📊 Your Daily Nutrition Summary – {now_ist().strftime('%d %b %Y')}",
            f"Hi {username}! You consumed {consumed:.0f} kcal today. Your target is {target:.0f} kcal.",
        )
        mail.add_html_message(html)
        mail.send()
        return True
    except Exception as e:
        st.warning(f"Email notification failed: {e}")
        return False

# ─────────────────────────────────────────────
# AUTH PAGE
# ─────────────────────────────────────────────
def render_auth():
    st.markdown("""
    <div style="text-align:center; padding: 40px 0 20px;">
        <div style="font-size:4rem;">🥗</div>
        <h1 style="background:linear-gradient(135deg,#63b3ed,#9a75ea,#f093fb);
                   -webkit-background-clip:text;-webkit-text-fill-color:transparent;
                   font-size:2.4rem;font-weight:800;margin:8px 0;">
            DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR
        </h1>
        <p style="color:#a0aec0;font-size:1rem;max-width:480px;margin:0 auto;">
            Your intelligent companion for achieving your health goals through smart nutrition tracking.
        </p>
    </div>
    """, unsafe_allow_html=True)

    col1, col2, col3 = st.columns([1, 2, 1])
    with col2:
        tab1, tab2 = st.tabs(["🔐 Login", "✨ Sign Up"])

        with tab1:
            st.markdown("<br>", unsafe_allow_html=True)
            with st.form("login_form"):
                email = st.text_input("📧 Email Address", placeholder="you@example.com")
                password = st.text_input("🔒 Password", type="password", placeholder="Enter your password")
                submit = st.form_submit_button("Login →", use_container_width=True)
                if submit:
                    if not email or not password:
                        st.error("Please fill in all fields.")
                    else:
                        with st.spinner("Authenticating..."):
                            ok, msg = login_user(email, password)
                        if ok:
                            st.success(msg)
                            st.rerun()
                        else:
                            st.error(msg)

        with tab2:
            st.markdown("<br>", unsafe_allow_html=True)
            with st.form("signup_form"):
                name = st.text_input("👤 Full Name", placeholder="John Doe")
                email_s = st.text_input("📧 Email Address", placeholder="you@example.com")
                pw1 = st.text_input("🔒 Password", type="password", placeholder="Min 6 characters")
                pw2 = st.text_input("🔒 Confirm Password", type="password", placeholder="Re-enter password")
                submit_s = st.form_submit_button("Create Account →", use_container_width=True)
                if submit_s:
                    if not all([name, email_s, pw1, pw2]):
                        st.error("Please fill in all fields.")
                    elif pw1 != pw2:
                        st.error("Passwords do not match.")
                    elif len(pw1) < 6:
                        st.error("Password must be at least 6 characters.")
                    else:
                        with st.spinner("Creating your account..."):
                            ok, result = signup_user(name, email_s, pw1)
                        if ok:
                            st.success("Account created! Please login.")
                        else:
                            st.error(result)

    st.markdown("""
    <div style="display:flex;justify-content:center;gap:40px;margin-top:48px;flex-wrap:wrap;">
        <div style="text-align:center;">
            <div style="font-size:2rem;">🤖</div>
            <p style="color:#63b3ed;font-weight:600;margin:4px 0;">AI-Powered</p>
            <p style="color:#718096;font-size:0.8rem;">Smart food analysis</p>
        </div>
        <div style="text-align:center;">
            <div style="font-size:2rem;">📊</div>
            <p style="color:#9a75ea;font-weight:600;margin:4px 0;">Deep Analytics</p>
            <p style="color:#718096;font-size:0.8rem;">Visual progress tracking</p>
        </div>
        <div style="text-align:center;">
            <div style="font-size:2rem;">🔔</div>
            <p style="color:#f093fb;font-weight:600;margin:4px 0;">Smart Alerts</p>
            <p style="color:#718096;font-size:0.8rem;">Email notifications</p>
        </div>
        <div style="text-align:center;">
            <div style="font-size:2rem;">🎯</div>
            <p style="color:#48c78e;font-weight:600;margin:4px 0;">Goal Tracking</p>
            <p style="color:#718096;font-size:0.8rem;">Personalized targets</p>
        </div>
    </div>
    """, unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SECTION 1: HOME
# ─────────────────────────────────────────────
def render_home():
    st.markdown('<h1 class="section-header">🏠 Welcome to DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR </h1>', unsafe_allow_html=True)
    st.markdown(f"<p style='color:#a0aec0;'>Hello, <strong style='color:#63b3ed;'>{st.session_state.username}</strong>! Ready to crush your health goals today? 💪</p>", unsafe_allow_html=True)

    st.markdown("""
---

## 🌟 What is DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR?

**DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR** is an intelligent, all-in-one platform that transforms how you think about food and health. We combine cutting-edge artificial intelligence with personalized nutrition science to give you insights that no ordinary calorie app can match.

---

## 🚀 How It Solves Your Biggest Nutrition Challenges

### 📸 Effortless Food Logging
- **No manual database searching** — just upload a photo of your meal or describe it in plain text
- Automatically detects every food item, estimates portions, and calculates complete nutrition in seconds
- Supports Indian cuisine, international dishes, homemade food, and complex mixed meals

### 🧠 Intelligent, Personalized Insights
- Goes beyond simple calorie counting — understands your **unique body, goals, and lifestyle**
- Generates actionable health recommendations tailored specifically to you
- Identifies nutritional gaps and suggests real foods to address them

### 📊 Beautiful Visual Analytics
- See your nutrition journey through stunning, interactive charts
- Track macro trends, calorie patterns, and progress toward your goals over time
- Weekly and monthly breakdowns with color-coded alerts

### 🔔 Proactive Health Alerts
- Get notified the moment you're about to exceed your daily calorie target
- Daily summary reports delivered straight to your inbox
- Stay accountable even when life gets busy

---

## 💎 What Makes Us Uniquely Different

| Feature | Ordinary Apps | DATA-DRIVEN SMART FOOD ANALYTICS AND NUTRITION ADVISOR |
|---|---|---|
| Food Detection | Manual search | **Photo / Text → Instant AI detection** |
| Insights | Generic tips | **Personalized to your profile & goals** |
| Portion Adjustment | Fixed values | **Interactive sliders with live recalculation** |
| Notifications | None | **Smart calorie alerts + daily summaries** |
| Analytics | Basic charts | **Deep insights with trend analysis** |
| Indian Food Support | ❌ Limited | **✅ Fully supported** |

---

## 🗺️ How to Get Started

1. **📋 User Information** → Fill in your health profile to unlock personalized targets
2. **🥘 AI Tracker** → Log meals via photo or text and get instant nutrition analysis
3. **📈 Analytics** → Track your progress and discover patterns in your eating habits

---

## ❓ Frequently Asked Questions
"""  )

    faq_items = [
        (
            "🎯 How accurate is the nutrition data?",
            "Our intelligent analysis uses comprehensive nutrition databases and real-world portion size data. Accuracy is typically within 10–15% of actual values — comparable to what a professional nutritionist would estimate for home-cooked meals. For best accuracy, adjust the quantity sliders after detection.",
        ),
        (
            "⚖️ Can I adjust portion sizes after detection?",
            "Absolutely! After food detection, every identified item comes with an interactive quantity slider. You can fine-tune each portion — all nutrition totals recalculate instantly on the fly without any extra AI call.",
        ),
        (
            "🔒 Is my data private and secure?",
            "Yes, completely. Your data is stored securely in your personal account and is never shared with third parties. Only you can access your nutrition history, health profile, and meal logs.",
        ),
        (
            "🍛 What cuisines and foods are supported?",
            "We support all cuisines without limitation — Indian (North, South, East, West regional dishes), Mediterranean, Asian, Western, homemade meals, street food, and packaged products. Our AI recognizes both complex multi-ingredient dishes and individual raw ingredients.",
        ),
        (
            "📧 How do email alerts and notifications work?",
            "After every meal is saved, we automatically check your total intake for that day. If it exceeds your personalized daily target, an alert email is sent immediately with insights. You can also manually trigger a full daily summary report anytime from the Analytics section.",
        ),
        (
            "📊 What kind of analytics will I see?",
            "The Analytics dashboard shows calorie trends over time, macro breakdowns (protein/carbs/fats), meal-type distribution, health scores per meal, fiber intake, and a daily gap analysis — all with interactive Plotly charts and a color-coded summary table.",
        ),
    ]

    for question, answer in faq_items:
        with st.expander(question):
            st.markdown(f"<p style='color:#e2e8f0;line-height:1.7;'>{answer}</p>", unsafe_allow_html=True)

    st.markdown("""
---
> 💡 **Pro Tip:** For best results, complete your **User Information** profile first. The more we know about you, the more accurate and personalized your insights will be!
""")

# ─────────────────────────────────────────────
# SECTION 2: USER INFORMATION
# ─────────────────────────────────────────────
def render_user_info():
    st.markdown('<h1 class="section-header">👤 User Information & Health Profile</h1>', unsafe_allow_html=True)
    st.markdown("<p style='color:#a0aec0;'>Complete your health profile to unlock personalized AI insights and accurate calorie targets.</p>", unsafe_allow_html=True)

    # Load existing profile
    existing = profiles_col.find_one({"user_id": st.session_state.user_id}) or {}

    with st.form("user_profile_form"):
        # ── Basic Info ──
        st.markdown("### 🧍 Basic Information")
        c1, c2, c3 = st.columns(3)
        age    = c1.number_input("Age (years)", 10, 100, int(existing.get("age", 25)))
        gender = c2.selectbox("Gender", ["Male", "Female", "Other"], index=["Male","Female","Other"].index(existing.get("gender","Male")))
        weight = c3.number_input("Weight (kg)", 20.0, 300.0, float(existing.get("weight_kg", 70.0)), step=0.5)

        c4, c5 = st.columns(2)
        height = c4.number_input("Height (cm)", 100.0, 250.0, float(existing.get("height_cm", 170.0)), step=0.5)
        bmi_val = weight / ((height/100)**2)
        c5.metric("Your BMI", f"{bmi_val:.1f}", delta=None)

        st.divider()

        # ── Activity & Goals ──
        st.markdown("### 🏃 Activity & Goals")
        c6, c7 = st.columns(2)
        activity = c6.selectbox("Activity Level", [
            "Sedentary (desk job, little exercise)",
            "Lightly Active (1–3 days/week exercise)",
            "Moderately Active (3–5 days/week exercise)",
            "Very Active (6–7 days/week exercise)",
            "Extremely Active (twice/day training)",
        ], index=["Sedentary (desk job, little exercise)","Lightly Active (1–3 days/week exercise)","Moderately Active (3–5 days/week exercise)","Very Active (6–7 days/week exercise)","Extremely Active (twice/day training)"].index(existing.get("activity_level", "Moderately Active (3–5 days/week exercise)")))

        goal = c7.selectbox("Health Goal", [
            "Lose Weight", "Maintain Weight", "Gain Muscle", "Improve Endurance",
            "Improve Overall Health", "Manage a Medical Condition",
        ], index=["Lose Weight","Maintain Weight","Gain Muscle","Improve Endurance","Improve Overall Health","Manage a Medical Condition"].index(existing.get("goal","Maintain Weight")))

        st.divider()

        # ── Diet & Health ──
        st.markdown("### 🥦 Diet & Health")
        c8, c9 = st.columns(2)
        diet_type = c8.selectbox("Dietary Preference", [
            "No Restriction (Non-Vegetarian)", "Vegetarian", "Vegan",
            "Pescatarian", "Keto", "Low-Carb", "Gluten-Free",
        ], index=["No Restriction (Non-Vegetarian)","Vegetarian","Vegan","Pescatarian","Keto","Low-Carb","Gluten-Free"].index(existing.get("diet_type","No Restriction (Non-Vegetarian)")))
        sleep_hours = c9.slider("Sleep Hours per Day", 4, 12, int(existing.get("sleep_hours", 7)))

        medical_conditions = st.text_area("Medical Conditions (if any)", value=existing.get("medical_conditions",""), placeholder="e.g., Diabetes Type 2, Hypertension, PCOS, Thyroid... or type 'None'")
        allergies = st.text_area("Food Allergies / Intolerances", value=existing.get("allergies",""), placeholder="e.g., Lactose intolerant, Nut allergy... or type 'None'")

        st.divider()

        # ── Daily Calorie Target ──
        st.markdown("### 🎯 Daily Calorie Target")
        st.info("💡 Click **Get AI Insights** first to get your recommended target, then set it below.")
        daily_cal = st.number_input(
            "Your Daily Calorie Target (kcal)",
            min_value=500, max_value=6000,
            value=int(existing.get("daily_calorie_target", 2000)),
            step=50,
            help="This is used to track whether you've hit your daily limit and trigger email alerts."
        )

        col_save, col_insights = st.columns([1, 1])
        save_btn = col_save.form_submit_button("💾 Save Profile", use_container_width=True)
        insights_btn = col_insights.form_submit_button("🤖 Get AI Insights", use_container_width=True)

        if save_btn or insights_btn:
            profile_data = {
                "user_id": st.session_state.user_id,
                "name": st.session_state.username,
                "age": age,
                "gender": gender,
                "weight_kg": weight,
                "height_cm": height,
                "bmi": round(bmi_val, 1),
                "activity_level": activity,
                "goal": goal,
                "diet_type": diet_type,
                "sleep_hours": sleep_hours,
                "medical_conditions": medical_conditions or "None",
                "allergies": allergies or "None",
                "daily_calorie_target": daily_cal,
                "updated_at": now_ist().isoformat(),
            }
            profiles_col.update_one(
                {"user_id": st.session_state.user_id},
                {"$set": profile_data},
                upsert=True
            )
            st.session_state.profile_complete = True
            st.session_state["cached_profile"] = profile_data

            if save_btn:
                st.success("✅ Profile saved successfully!")

            if insights_btn:
                with st.spinner("🤖 Generating your personalized AI insights..."):
                    insights_text = get_profile_insights(profile_data)
                st.session_state["profile_insights"] = insights_text

    # Show AI insights outside form
    if st.session_state.get("profile_insights"):
        st.markdown("---")
        st.markdown("## 🤖 Your Personalized AI Health Insights")
        st.markdown(
            f'<div class="glass-card">{st.session_state["profile_insights"]}</div>',
            unsafe_allow_html=True
        )

# ─────────────────────────────────────────────
# SECTION 3: AI TRACKER
# ─────────────────────────────────────────────
def render_tracker():
    if not st.session_state.profile_complete:
        st.markdown("""
        <div class="warning-banner">
            ⚠️ <strong>Profile Required</strong> — Please complete your health profile in <strong>User Information</strong> section first to unlock the AI Tracker.
        </div>
        """, unsafe_allow_html=True)
        return

    st.markdown('<h1 class="section-header">🥘 AI Food Tracker</h1>', unsafe_allow_html=True)
    st.markdown("<p style='color:#a0aec0;'>Upload a meal photo or describe your food — our AI will analyze nutrition instantly.</p>", unsafe_allow_html=True)

    profile = profiles_col.find_one({"user_id": st.session_state.user_id}) or {}

    # ── Input Form ──
    with st.container():
        st.markdown('<div class="glass-card">', unsafe_allow_html=True)
        st.markdown("### 📝 Meal Details")

        c1, c2 = st.columns(2)
        meal_type = c1.selectbox("🍽️ Meal Type", ["Breakfast", "Lunch", "Dinner", "Snack", "Pre-Workout", "Post-Workout", "Other"])
        meal_date = c2.date_input("📅 Date", value=now_ist().date())

        c3, c4 = st.columns(2)
        meal_time = c3.time_input("⏰ Time", value=now_ist().time())
        location  = c4.text_input("📍 Location (optional)", placeholder="e.g., Home, Restaurant, Office")

        notes = st.text_input("📌 Notes (optional)", placeholder="e.g., Cheat day, Post workout, Birthday dinner")

        st.divider()
        st.markdown("### 🔍 Food Input")
        input_mode = st.radio("Input Method", ["📸 Upload Image", "✍️ Text Description"], horizontal=True)

        food_text  = None
        image_bytes = None
        mime_type   = "image/jpeg"

        if "📸 Upload Image" in input_mode:
            uploaded = st.file_uploader("Upload meal photo", type=["jpg","jpeg","png","webp"], label_visibility="collapsed")
            if uploaded:
                st.image(uploaded, caption="Your meal", width=350)
                image_bytes = uploaded.read()
                mime_type = uploaded.type or "image/jpeg"
        else:
            food_text = st.text_area(
                "Describe your meal",
                placeholder="e.g., 2 chapati with dal makhani and a glass of buttermilk, one banana",
                height=100,
            )

        detect_btn = st.button("🔍 Analyse Meal", use_container_width=True)
        st.markdown('</div>', unsafe_allow_html=True)

    if detect_btn:
        if input_mode == "📸 Upload Image" and not image_bytes:
            st.error("Please upload an image first.")
        elif "✍️ Text Description" in input_mode and not food_text:
            st.error("Please describe your meal.")
        else:
            with st.spinner("🤖 AI is analysing your meal..."):
                try:
                    if image_bytes:
                        result = detect_foods_from_image(image_bytes, mime_type)
                    else:
                        result = detect_foods_from_text(food_text)
                    st.session_state.food_detection_result = result
                    st.session_state.quantities_dict = {f.food_name: f.default_quantity for f in result.foods}
                    st.session_state.tracker_totals = None
                    st.session_state.tracker_insights = None
                    st.session_state.tracker_submitted = False
                    st.session_state["tracker_meta"] = {
                        "meal_type": meal_type,
                        "meal_date": meal_date.isoformat(),
                        "meal_time": meal_time.strftime("%H:%M"),
                        "location": location,
                        "notes": notes,
                        "input_mode": "image" if image_bytes else "text",
                        "food_text": food_text,
                    }
                except Exception as e:
                    st.error(f"AI analysis failed: {e}")

    # ── Results ──
    if st.session_state.food_detection_result:
        result = st.session_state.food_detection_result
        st.markdown("---")
        st.markdown("## 🥗 Detected Foods — Adjust Your Quantities")
        st.info("💡 Use the sliders to adjust portion sizes. Nutrition totals update automatically.")

        updated_quantities = {}
        cols_per_row = 2
        food_list = result.foods
        for i in range(0, len(food_list), cols_per_row):
            cols = st.columns(cols_per_row)
            for j, food in enumerate(food_list[i:i+cols_per_row]):
                with cols[j]:
                    with st.container():
                        st.markdown(f"""
                        <div class="glass-card" style="margin-bottom:8px;">
                            <h4 style="color:#63b3ed;margin:0 0 4px;">🍴 {food.food_name}</h4>
                            <p style="color:#a0aec0;font-size:0.8rem;margin:0;">
                                Per {food.nutrients_per_unit.unit}:
                                <strong>{food.nutrients_per_unit.calories_kcal:.0f}</strong> kcal |
                                P: <strong>{food.nutrients_per_unit.protein_g:.1f}g</strong> |
                                C: <strong>{food.nutrients_per_unit.carbs_g:.1f}g</strong> |
                                F: <strong>{food.nutrients_per_unit.fats_g:.1f}g</strong>
                            </p>
                        </div>
                        """, unsafe_allow_html=True)
                        qty = st.slider(
                            f"Quantity ({food.nutrients_per_unit.unit})",
                            min_value=0.0,
                            max_value=max(20.0, food.default_quantity * 3),
                            value=float(st.session_state.quantities_dict.get(food.food_name, food.default_quantity)),
                            step=0.5,
                            key=f"qty_{food.food_name}_{i}_{j}",
                        )
                        updated_quantities[food.food_name] = qty

        st.session_state.quantities_dict = updated_quantities

        # Live totals
        totals = calculate_totals(result.foods, updated_quantities)
        st.session_state.tracker_totals = totals

        st.markdown("---")
        st.markdown("## 📊 Nutrition Totals")
        tc1, tc2, tc3, tc4, tc5 = st.columns(5)
        tc1.markdown(f'<div class="stat-card"><div class="metric-value">{totals["calories_kcal"]:.0f}</div><div class="metric-label">🔥 Calories (kcal)</div></div>', unsafe_allow_html=True)
        tc2.markdown(f'<div class="stat-card"><div class="metric-value">{totals["protein_g"]:.1f}g</div><div class="metric-label">💪 Protein</div></div>', unsafe_allow_html=True)
        tc3.markdown(f'<div class="stat-card"><div class="metric-value">{totals["carbs_g"]:.1f}g</div><div class="metric-label">🌾 Carbs</div></div>', unsafe_allow_html=True)
        tc4.markdown(f'<div class="stat-card"><div class="metric-value">{totals["fats_g"]:.1f}g</div><div class="metric-label">🥑 Fats</div></div>', unsafe_allow_html=True)
        tc5.markdown(f'<div class="stat-card"><div class="metric-value">{totals["fiber_g"]:.1f}g</div><div class="metric-label">🌿 Fiber</div></div>', unsafe_allow_html=True)

        # Macro pie chart
        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(data=[go.Pie(
            labels=["Protein", "Carbs", "Fats", "Fiber"],
            values=[totals["protein_g"], totals["carbs_g"], totals["fats_g"], totals["fiber_g"]],
            hole=0.55,
            marker_colors=["#9a75ea","#63b3ed","#f6ad55","#48c78e"],
        )])
        fig.update_layout(
            title="Macro Distribution",
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,0,0,0)",
            font_color="#e2e8f0",
            showlegend=True,
            height=300,
            margin=dict(t=40, b=0, l=0, r=0),
        )
        st.plotly_chart(fig, use_container_width=True)

        # Submit button
        if not st.session_state.tracker_submitted:
            if st.button("✅ Get AI Insights & Save Meal", use_container_width=True):
                with st.spinner("🤖 Generating AI health insights for this meal..."):
                    try:
                        insights = get_ai_insights(totals, profile, st.session_state["tracker_meta"]["meal_type"], result.foods)
                        st.session_state.tracker_insights = insights
                    except Exception as e:
                        st.error(f"Insights generation failed: {e}")
                        return

                # Save to DB
                meta = st.session_state.get("tracker_meta", {})
                meal_doc = {
                    "user_id": st.session_state.user_id,
                    "meal_type": meta.get("meal_type"),
                    "meal_date": meta.get("meal_date"),
                    "meal_time": meta.get("meal_time"),
                    "location": meta.get("location",""),
                    "notes": meta.get("notes",""),
                    "input_mode": meta.get("input_mode"),
                    "food_description": meta.get("food_text",""),
                    "foods": [f.model_dump() for f in result.foods],
                    "quantities_used": updated_quantities,
                    "final_totals": totals,
                    "insights": {
                        "summary": insights.summary,
                        "health_score": insights.health_score,
                        "suggestions": insights.suggestions,
                        "positives": insights.positives,
                        "warnings": insights.warnings,
                    },
                    "saved_at": now_ist().isoformat(),
                }
                meals_col.insert_one(meal_doc)
                st.session_state.tracker_submitted = True

                # Check daily calorie limit
                today_str = meta.get("meal_date", now_ist().date().isoformat())
                today_meals = list(meals_col.find({"user_id": st.session_state.user_id, "meal_date": today_str}))
                today_total = sum(m.get("final_totals", {}).get("calories_kcal", 0) for m in today_meals)
                daily_target = profile.get("daily_calorie_target", 2000)

                if today_total > daily_target:
                    st.markdown(f"""
                    <div class="warning-banner">
                        🚨 <strong>Calorie Alert!</strong> You've consumed <strong>{today_total:.0f} kcal</strong> today,
                        which exceeds your daily target of <strong>{daily_target:.0f} kcal</strong>
                        by <strong>{today_total - daily_target:.0f} kcal</strong>. Sending email alert...
                    </div>
                    """, unsafe_allow_html=True)
                    send_calorie_alert_email(
                        st.session_state.user_email,
                        st.session_state.username,
                        today_total, daily_target, insights
                    )

                st.success("✅ Meal saved successfully!")
                st.rerun()

        # Show insights
        if st.session_state.tracker_insights:
            insights = st.session_state.tracker_insights
            st.markdown("---")
            st.markdown("## 🤖 AI Meal Analysis")

            score = insights.health_score
            score_color = "#48c78e" if score >= 7 else "#f6ad55" if score >= 5 else "#ef4444"
            score_label = "Excellent" if score >= 8 else "Good" if score >= 6 else "Fair" if score >= 4 else "Needs Improvement"

            # Meal summary card
            st.markdown(f"""
            <div class="glass-card">
                <div style="display:flex;align-items:center;gap:16px;flex-wrap:wrap;">
                    <div style="text-align:center;min-width:80px;">
                        <div style="font-size:2.4rem;font-weight:800;color:{score_color};">{score:.0f}<span style="font-size:1rem;color:#718096;">/10</span></div>
                        <div style="font-size:0.75rem;color:{score_color};font-weight:600;text-transform:uppercase;letter-spacing:0.05em;">{score_label}</div>
                    </div>
                    <div style="flex:1;min-width:200px;">
                        <p style="color:#ffffff;font-size:1rem;line-height:1.6;margin:0;font-weight:400;">{insights.summary}</p>
                    </div>
                </div>
            </div>
            """, unsafe_allow_html=True)

            # Three columns: benefits | goal alignment (suggestions) | warnings
            ci1, ci2, ci3 = st.columns(3)

            with ci1:
                st.markdown("""
                <div style="background:rgba(72,199,142,0.08);border:1px solid rgba(72,199,142,0.25);
                            border-radius:12px;padding:16px;height:100%;">
                    <h4 style="color:#48c78e;margin:0 0 12px;font-size:0.95rem;display:flex;align-items:center;gap:6px;">
                        ✅ Health Benefits
                    </h4>
                """, unsafe_allow_html=True)
                for p in insights.positives:
                    st.markdown(f"<p style='color:#e2e8f0;font-size:0.85rem;line-height:1.5;margin:0 0 8px;padding-left:4px;border-left:2px solid #48c78e;'>&#x2022; {p}</p>", unsafe_allow_html=True)
                st.markdown("</div>", unsafe_allow_html=True)

            with ci2:
                goal_text = profile.get('goal', 'your goal')
                st.markdown(f"""
                <div style="background:rgba(99,179,237,0.08);border:1px solid rgba(99,179,237,0.25);
                            border-radius:12px;padding:16px;height:100%;">
                    <h4 style="color:#63b3ed;margin:0 0 12px;font-size:0.95rem;display:flex;align-items:center;gap:6px;">
                        🎯 Goal Alignment <span style="font-size:0.75rem;color:#718096;">({goal_text})</span>
                    </h4>
                """, unsafe_allow_html=True)
                for s in insights.suggestions:
                    st.markdown(f"<p style='color:#e2e8f0;font-size:0.85rem;line-height:1.5;margin:0 0 8px;padding-left:4px;border-left:2px solid #63b3ed;'>&#x2022; {s}</p>", unsafe_allow_html=True)
                st.markdown("</div>", unsafe_allow_html=True)

            with ci3:
                st.markdown("""
                <div style="background:rgba(246,173,85,0.08);border:1px solid rgba(246,173,85,0.25);
                            border-radius:12px;padding:16px;height:100%;">
                    <h4 style="color:#f6ad55;margin:0 0 12px;font-size:0.95rem;display:flex;align-items:center;gap:6px;">
                        ⚠️ Watch Out
                    </h4>
                """, unsafe_allow_html=True)
                for w in insights.warnings:
                    st.markdown(f"<p style='color:#e2e8f0;font-size:0.85rem;line-height:1.5;margin:0 0 8px;padding-left:4px;border-left:2px solid #f6ad55;'>&#x2022; {w}</p>", unsafe_allow_html=True)
                st.markdown("</div>", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SECTION 4: ANALYTICS
# ─────────────────────────────────────────────

CHART_LAYOUT = dict(
    paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
    font_color="#e2e8f0", margin=dict(t=30, b=20, l=10, r=10),
    xaxis=dict(gridcolor="rgba(255,255,255,0.07)", tickfont=dict(color="#e2e8f0")),
    yaxis=dict(gridcolor="rgba(255,255,255,0.07)", tickfont=dict(color="#e2e8f0")),
    legend=dict(font=dict(color="#e2e8f0")),
)

def kpi_cards(n_meals, avg_cal, total_protein, avg_score, days_breached):
    score_color = "#48c78e" if avg_score >= 7 else "#f6ad55" if avg_score >= 5 else "#ef4444"
    breach_color = "#ef4444" if days_breached > 0 else "#48c78e"
    cols = st.columns(5)
    cols[0].markdown(f'<div class="stat-card"><div class="metric-value">{n_meals}</div><div class="metric-label">Total Meals</div></div>', unsafe_allow_html=True)
    cols[1].markdown(f'<div class="stat-card"><div class="metric-value">{avg_cal:.0f}</div><div class="metric-label">Avg Daily kcal</div></div>', unsafe_allow_html=True)
    cols[2].markdown(f'<div class="stat-card"><div class="metric-value">{total_protein:.0f}g</div><div class="metric-label">Total Protein</div></div>', unsafe_allow_html=True)
    cols[3].markdown(f'<div class="stat-card"><div class="metric-value" style="color:{score_color}">{avg_score:.1f}</div><div class="metric-label">Avg Health Score</div></div>', unsafe_allow_html=True)
    cols[4].markdown(f'<div class="stat-card"><div class="metric-value" style="color:{breach_color}">{days_breached}</div><div class="metric-label">Days Over Target</div></div>', unsafe_allow_html=True)

def color_status_rows(daily_target):
    def _fn(row):
        cal = float(str(row.get("Total Calories", row.get("Avg Calories", 0))).replace(",",""))
        color = "background-color: rgba(239,68,68,0.18); color: #ef4444" if cal > daily_target else "background-color: rgba(72,199,142,0.12); color: #48c78e"
        return [color if col == "Status" else "" for col in row.index]
    return _fn

def render_analytics():
    if not st.session_state.profile_complete:
        st.markdown('<div class="warning-banner">⚠️ <strong>Profile Required</strong> — Please complete your health profile in <strong>User Information</strong> first to unlock Analytics.</div>', unsafe_allow_html=True)
        return

    st.markdown('<h1 class="section-header">📈 Nutrition Analytics Dashboard</h1>', unsafe_allow_html=True)

    profile = profiles_col.find_one({"user_id": st.session_state.user_id}) or {}
    daily_target = profile.get("daily_calorie_target", 2000)

    all_meals = list(meals_col.find({"user_id": st.session_state.user_id}).sort("meal_date", 1))
    if not all_meals:
        st.info("🍽️ No meals logged yet. Start tracking in the **AI Tracker** section!")
        return

    # ── Build master DataFrame ──
    rows = []
    for m in all_meals:
        rows.append({
            "Date": m.get("meal_date", ""),
            "Time": m.get("meal_time", ""),
            "Meal Type": m.get("meal_type", ""),
            "Calories": m.get("final_totals", {}).get("calories_kcal", 0),
            "Protein":  m.get("final_totals", {}).get("protein_g", 0),
            "Carbs":    m.get("final_totals", {}).get("carbs_g", 0),
            "Fats":     m.get("final_totals", {}).get("fats_g", 0),
            "Fiber":    m.get("final_totals", {}).get("fiber_g", 0),
            "Health Score": m.get("insights", {}).get("health_score", None),
            "Location": m.get("location", ""),
            "Notes":    m.get("notes", ""),
        })
    df = pd.DataFrame(rows)
    df["Date"] = pd.to_datetime(df["Date"])
    df["Week"]  = df["Date"].dt.to_period("W").apply(lambda r: r.start_time)
    df["Month"] = df["Date"].dt.to_period("M").apply(lambda r: r.start_time)
    df["WeekLabel"]  = df["Date"].dt.to_period("W").apply(lambda r: f"W{r.week} ({r.start_time.strftime('%d %b')}–{r.end_time.strftime('%d %b')})")
    df["MonthLabel"] = df["Date"].dt.to_period("M").apply(lambda r: r.start_time.strftime("%b %Y"))

    # ── Global Filters ──
    st.markdown("### 🔎 Global Filters")
    fc1, fc2, fc3 = st.columns(3)
    min_date, max_date = df["Date"].min().date(), df["Date"].max().date()
    date_range = fc1.date_input("Date Range", value=(min_date, max_date), min_value=min_date, max_value=max_date)
    meal_filter = fc2.multiselect("Meal Type", options=sorted(df["Meal Type"].unique().tolist()), default=sorted(df["Meal Type"].unique().tolist()))
    min_cal, max_cal = int(df["Calories"].min()), max(int(df["Calories"].max()) + 1, int(df["Calories"].min()) + 1)
    cal_range = fc3.slider("Calorie Range (per meal)", min_cal, max_cal, (min_cal, max_cal))

    if isinstance(date_range, (list, tuple)) and len(date_range) == 2:
        df_f = df[(df["Date"].dt.date >= date_range[0]) & (df["Date"].dt.date <= date_range[1])]
    else:
        df_f = df.copy()
    df_f = df_f[df_f["Meal Type"].isin(meal_filter)]
    df_f = df_f[(df_f["Calories"] >= cal_range[0]) & (df_f["Calories"] <= cal_range[1])]

    st.markdown("---")

    # ════════════════════════════════════════
    # TABS
    # ════════════════════════════════════════
    tab_daily, tab_weekly, tab_monthly = st.tabs(["📅 Daily View", "📆 Weekly View", "🗓️ Monthly View"])

    # ─────────────────────────────────
    # TAB 1 : DAILY
    # ─────────────────────────────────
    with tab_daily:
        daily_df = df_f.groupby("Date").agg(
            Calories=("Calories","sum"), Protein=("Protein","sum"),
            Carbs=("Carbs","sum"), Fats=("Fats","sum"),
            Fiber=("Fiber","sum"), Health_Score=("Health Score","mean"),
            Meals=("Meal Type","count"),
        ).reset_index()

        st.markdown("### 📊 Daily KPIs")
        breached = (daily_df["Calories"] > daily_target).sum()
        avg_score = df_f["Health Score"].mean() or 0
        kpi_cards(len(df_f), daily_df["Calories"].mean(), df_f["Protein"].sum(), avg_score, breached)

        st.markdown("---")
        st.markdown("### 🗓️ Daily Summary Table")
        st.caption("🟢 Green = Within target &nbsp;|&nbsp; 🔴 Red = Exceeded target")

        disp = daily_df.copy()
        disp["Date"] = disp["Date"].dt.strftime("%d %b %Y")
        disp["Status"] = daily_df["Calories"].apply(lambda x: "✅ On Track" if x <= daily_target else f"🚨 +{x-daily_target:.0f} kcal").values
        disp = disp.rename(columns={"Calories":"Total Calories","Protein":"Protein (g)","Carbs":"Carbs (g)","Fats":"Fats (g)","Fiber":"Fiber (g)","Health_Score":"Avg Score","Meals":"# Meals"})
        styled = disp.style.apply(color_status_rows(daily_target), axis=1).format({
            "Total Calories":"{:.0f}","Protein (g)":"{:.1f}","Carbs (g)":"{:.1f}",
            "Fats (g)":"{:.1f}","Fiber (g)":"{:.1f}","Avg Score":"{:.1f}",
        })
        st.dataframe(styled, use_container_width=True, height=300)

        st.markdown("---")

        # Calorie trend
        st.markdown("### 📉 Daily Calorie Trend")
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=daily_df["Date"], y=daily_df["Calories"],
            mode="lines+markers", name="Daily Calories",
            line=dict(color="#63b3ed", width=2.5),
            marker=dict(size=7, color=["#ef4444" if c > daily_target else "#48c78e" for c in daily_df["Calories"]]),
            fill="tozeroy", fillcolor="rgba(99,179,237,0.08)",
        ))
        fig.add_hline(y=daily_target, line_dash="dash", line_color="#f6ad55", annotation_text=f"Target: {daily_target} kcal", annotation_font_color="#f6ad55")
        fig.update_layout(**CHART_LAYOUT, height=320)
        st.plotly_chart(fig, use_container_width=True)

        g1, g2 = st.columns(2)
        with g1:
            st.markdown("#### 🥩 Daily Macro Breakdown")
            fig_m = go.Figure()
            fig_m.add_trace(go.Bar(x=daily_df["Date"], y=daily_df["Protein"], name="Protein", marker_color="#9a75ea"))
            fig_m.add_trace(go.Bar(x=daily_df["Date"], y=daily_df["Carbs"],   name="Carbs",   marker_color="#63b3ed"))
            fig_m.add_trace(go.Bar(x=daily_df["Date"], y=daily_df["Fats"],    name="Fats",    marker_color="#f6ad55"))
            fig_m.update_layout(**CHART_LAYOUT, barmode="stack", height=300)
            st.plotly_chart(fig_m, use_container_width=True)

        with g2:
            st.markdown("#### 🍽️ Calories by Meal Type")
            mc = df_f.groupby("Meal Type")["Calories"].sum().reset_index()
            fig_pie = px.pie(mc, values="Calories", names="Meal Type", hole=0.45,
                             color_discrete_sequence=["#63b3ed","#9a75ea","#48c78e","#f6ad55","#f093fb","#ef4444"])
            fig_pie.update_layout(paper_bgcolor="rgba(0,0,0,0)", font_color="#e2e8f0", height=300, margin=dict(t=20,b=20))
            st.plotly_chart(fig_pie, use_container_width=True)

        g3, g4 = st.columns(2)
        with g3:
            st.markdown("#### ⭐ Health Score Per Meal")
            sc = df_f.dropna(subset=["Health Score"]).copy()
            if not sc.empty:
                fig_s = go.Figure()
                fig_s.add_trace(go.Scatter(
                    x=sc["Date"], y=sc["Health Score"], mode="lines+markers",
                    line=dict(color="#48c78e", width=2),
                    marker=dict(size=8, color=["#48c78e" if s>=7 else "#f6ad55" if s>=5 else "#ef4444" for s in sc["Health Score"]]),
                ))
                fig_s.add_hline(y=7, line_dash="dot", line_color="#48c78e", annotation_text="Good (7)", annotation_font_color="#48c78e")
                fig_s.update_layout(**CHART_LAYOUT, height=280, yaxis_range=[0,10.5])
                st.plotly_chart(fig_s, use_container_width=True)

        with g4:
            st.markdown("#### 🥗 Daily Fiber Intake")
            fig_f = go.Figure()
            fig_f.add_trace(go.Bar(x=daily_df["Date"], y=daily_df["Fiber"], name="Fiber (g)", marker_color="#48c78e"))
            fig_f.add_hline(y=30, line_dash="dash", line_color="#f6ad55", annotation_text="Recommended: 30g", annotation_font_color="#f6ad55")
            fig_f.update_layout(**CHART_LAYOUT, height=280)
            st.plotly_chart(fig_f, use_container_width=True)

        st.markdown("#### 📊 Daily Calorie Gap (vs Target)")
        daily_df["Gap"] = daily_df["Calories"] - daily_target
        fig_gap = go.Figure(go.Bar(
            x=daily_df["Date"], y=daily_df["Gap"],
            marker_color=["#ef4444" if g > 0 else "#48c78e" for g in daily_df["Gap"]],
            text=[f"+{g:.0f}" if g>0 else f"{g:.0f}" for g in daily_df["Gap"]],
            textposition="outside", textfont=dict(color="#e2e8f0"),
        ))
        fig_gap.add_hline(y=0, line_color="#f6ad55", line_width=1.5)
        fig_gap.update_layout(**CHART_LAYOUT, height=280, yaxis_title="kcal vs target")
        st.plotly_chart(fig_gap, use_container_width=True)

    # ─────────────────────────────────
    # TAB 2 : WEEKLY
    # ─────────────────────────────────
    with tab_weekly:
        weekly_df = df_f.groupby("WeekLabel").agg(
            Calories=("Calories","sum"), Protein=("Protein","sum"),
            Carbs=("Carbs","sum"), Fats=("Fats","sum"),
            Fiber=("Fiber","sum"), Health_Score=("Health Score","mean"),
            Meals=("Meal Type","count"),
        ).reset_index()
        weekly_df["Avg Daily Cal"] = weekly_df["Calories"] / 7
        weekly_target = daily_target * 7

        st.markdown("### 📊 Weekly KPIs")
        w_breached = (weekly_df["Calories"] > weekly_target).sum()
        w_avg_score = df_f["Health Score"].mean() or 0
        kpi_cards(len(df_f), weekly_df["Calories"].mean()/7, df_f["Protein"].sum(), w_avg_score, w_breached)

        st.markdown("---")
        st.markdown("### 📋 Weekly Summary Table")
        st.caption("🟢 Green = Within weekly target &nbsp;|&nbsp; 🔴 Red = Exceeded weekly target")

        wdisp = weekly_df.copy()
        wdisp["Status"] = weekly_df["Calories"].apply(lambda x: "✅ On Track" if x <= weekly_target else f"🚨 +{x-weekly_target:.0f} kcal").values
        wdisp = wdisp.rename(columns={
            "WeekLabel":"Week","Calories":"Total Calories","Protein":"Protein (g)",
            "Carbs":"Carbs (g)","Fats":"Fats (g)","Fiber":"Fiber (g)",
            "Health_Score":"Avg Score","Meals":"# Meals","Avg Daily Cal":"Avg Daily kcal",
        })
        wdisp_cols = ["Week","Total Calories","Avg Daily kcal","Protein (g)","Carbs (g)","Fats (g)","Fiber (g)","Avg Score","# Meals","Status"]
        wstyled = wdisp[wdisp_cols].style.apply(color_status_rows(weekly_target), axis=1).format({
            "Total Calories":"{:.0f}","Avg Daily kcal":"{:.0f}","Protein (g)":"{:.1f}",
            "Carbs (g)":"{:.1f}","Fats (g)":"{:.1f}","Fiber (g)":"{:.1f}","Avg Score":"{:.1f}",
        })
        st.dataframe(wstyled, use_container_width=True, height=300)

        st.markdown("---")

        # Weekly calorie bar
        st.markdown("### 📉 Weekly Calorie Total")
        fig_wc = go.Figure()
        fig_wc.add_trace(go.Bar(
            x=weekly_df["WeekLabel"], y=weekly_df["Calories"],
            marker_color=["#ef4444" if c > weekly_target else "#63b3ed" for c in weekly_df["Calories"]],
            text=[f"{c:.0f}" for c in weekly_df["Calories"]], textposition="outside",
            textfont=dict(color="#e2e8f0"),
        ))
        fig_wc.add_hline(y=weekly_target, line_dash="dash", line_color="#f6ad55",
                         annotation_text=f"Weekly Target: {weekly_target:.0f} kcal", annotation_font_color="#f6ad55")
        fig_wc.update_layout(**CHART_LAYOUT, height=320)
        st.plotly_chart(fig_wc, use_container_width=True)

        wg1, wg2 = st.columns(2)
        with wg1:
            st.markdown("#### 🥩 Weekly Macro Totals")
            fig_wm = go.Figure()
            fig_wm.add_trace(go.Bar(x=weekly_df["WeekLabel"], y=weekly_df["Protein"], name="Protein", marker_color="#9a75ea"))
            fig_wm.add_trace(go.Bar(x=weekly_df["WeekLabel"], y=weekly_df["Carbs"],   name="Carbs",   marker_color="#63b3ed"))
            fig_wm.add_trace(go.Bar(x=weekly_df["WeekLabel"], y=weekly_df["Fats"],    name="Fats",    marker_color="#f6ad55"))
            fig_wm.update_layout(**CHART_LAYOUT, barmode="group", height=300)
            st.plotly_chart(fig_wm, use_container_width=True)

        with wg2:
            st.markdown("#### ⭐ Avg Health Score per Week")
            fig_ws = go.Figure()
            fig_ws.add_trace(go.Scatter(
                x=weekly_df["WeekLabel"], y=weekly_df["Health_Score"],
                mode="lines+markers+text", name="Avg Score",
                line=dict(color="#48c78e", width=2.5),
                marker=dict(size=9, color=["#48c78e" if s>=7 else "#f6ad55" if s>=5 else "#ef4444" for s in weekly_df["Health_Score"].fillna(0)]),
                text=[f"{s:.1f}" if pd.notna(s) else "" for s in weekly_df["Health_Score"]],
                textposition="top center", textfont=dict(color="#e2e8f0"),
            ))
            fig_ws.add_hline(y=7, line_dash="dot", line_color="#48c78e", annotation_text="Target (7)", annotation_font_color="#48c78e")
            fig_ws.update_layout(**CHART_LAYOUT, height=300, yaxis_range=[0,10.5])
            st.plotly_chart(fig_ws, use_container_width=True)

        wg3, wg4 = st.columns(2)
        with wg3:
            st.markdown("#### 🥗 Weekly Fiber Intake")
            fig_wf = go.Figure()
            fig_wf.add_trace(go.Bar(x=weekly_df["WeekLabel"], y=weekly_df["Fiber"], name="Fiber (g)",
                                    marker_color=["#48c78e" if f >= 210 else "#f6ad55" for f in weekly_df["Fiber"]]))
            fig_wf.add_hline(y=210, line_dash="dash", line_color="#f6ad55", annotation_text="Weekly Rec: 210g", annotation_font_color="#f6ad55")
            fig_wf.update_layout(**CHART_LAYOUT, height=280)
            st.plotly_chart(fig_wf, use_container_width=True)

        with wg4:
            st.markdown("#### 🍽️ Avg Meals per Week")
            fig_wn = go.Figure(go.Bar(
                x=weekly_df["WeekLabel"], y=weekly_df["Meals"],
                marker_color="#f093fb",
                text=weekly_df["Meals"], textposition="outside", textfont=dict(color="#e2e8f0"),
            ))
            fig_wn.update_layout(**CHART_LAYOUT, height=280, yaxis_title="Meals logged")
            st.plotly_chart(fig_wn, use_container_width=True)

        # Weekly calorie gap
        st.markdown("#### 📊 Weekly Calorie Gap (vs Target)")
        weekly_df["Gap"] = weekly_df["Calories"] - weekly_target
        fig_wgap = go.Figure(go.Bar(
            x=weekly_df["WeekLabel"], y=weekly_df["Gap"],
            marker_color=["#ef4444" if g > 0 else "#48c78e" for g in weekly_df["Gap"]],
            text=[f"+{g:.0f}" if g>0 else f"{g:.0f}" for g in weekly_df["Gap"]],
            textposition="outside", textfont=dict(color="#e2e8f0"),
        ))
        fig_wgap.add_hline(y=0, line_color="#f6ad55", line_width=1.5)
        fig_wgap.update_layout(**CHART_LAYOUT, height=280, yaxis_title="kcal vs weekly target")
        st.plotly_chart(fig_wgap, use_container_width=True)

    # ─────────────────────────────────
    # TAB 3 : MONTHLY
    # ─────────────────────────────────
    with tab_monthly:
        monthly_df = df_f.groupby("MonthLabel").agg(
            Calories=("Calories","sum"), Protein=("Protein","sum"),
            Carbs=("Carbs","sum"), Fats=("Fats","sum"),
            Fiber=("Fiber","sum"), Health_Score=("Health Score","mean"),
            Meals=("Meal Type","count"),
        ).reset_index()
        # Days in each month for avg
        monthly_days = df_f.groupby("MonthLabel")["Date"].apply(lambda x: x.dt.date.nunique()).reset_index()
        monthly_days.columns = ["MonthLabel", "Days"]
        monthly_df = monthly_df.merge(monthly_days, on="MonthLabel", how="left")
        monthly_df["Avg Daily Cal"] = monthly_df["Calories"] / monthly_df["Days"]
        monthly_target = daily_target * 30

        st.markdown("### 📊 Monthly KPIs")
        m_breached = (monthly_df["Avg Daily Cal"] > daily_target).sum()
        m_avg_score = df_f["Health Score"].mean() or 0
        kpi_cards(len(df_f), monthly_df["Avg Daily Cal"].mean(), df_f["Protein"].sum(), m_avg_score, m_breached)

        st.markdown("---")
        st.markdown("### 📋 Monthly Summary Table")
        st.caption("🟢 Green = Avg daily within target &nbsp;|&nbsp; 🔴 Red = Avg daily exceeded target")

        mdisp = monthly_df.copy()
        mdisp["Status"] = monthly_df["Avg Daily Cal"].apply(lambda x: "✅ On Track" if x <= daily_target else f"🚨 +{x-daily_target:.0f} kcal/day").values
        mdisp = mdisp.rename(columns={
            "MonthLabel":"Month","Calories":"Total Calories","Protein":"Protein (g)",
            "Carbs":"Carbs (g)","Fats":"Fats (g)","Fiber":"Fiber (g)",
            "Health_Score":"Avg Score","Meals":"# Meals","Avg Daily Cal":"Avg Daily kcal","Days":"Days Tracked",
        })
        mcols = ["Month","Total Calories","Avg Daily kcal","Days Tracked","Protein (g)","Carbs (g)","Fats (g)","Fiber (g)","Avg Score","# Meals","Status"]
        mstyled = mdisp[mcols].style.apply(color_status_rows(daily_target), axis=1).format({
            "Total Calories":"{:.0f}","Avg Daily kcal":"{:.0f}","Protein (g)":"{:.1f}",
            "Carbs (g)":"{:.1f}","Fats (g)":"{:.1f}","Fiber (g)":"{:.1f}","Avg Score":"{:.1f}",
        })
        st.dataframe(mstyled, use_container_width=True, height=300)

        st.markdown("---")

        # Monthly calorie bar
        st.markdown("### 📉 Monthly Total Calories")
        fig_mc = go.Figure()
        fig_mc.add_trace(go.Bar(
            x=monthly_df["MonthLabel"], y=monthly_df["Calories"],
            marker_color=["#ef4444" if c > monthly_target else "#63b3ed" for c in monthly_df["Calories"]],
            text=[f"{c:.0f}" for c in monthly_df["Calories"]], textposition="outside",
            textfont=dict(color="#e2e8f0"),
        ))
        fig_mc.update_layout(**CHART_LAYOUT, height=320, yaxis_title="Total kcal")
        st.plotly_chart(fig_mc, use_container_width=True)

        mg1, mg2 = st.columns(2)
        with mg1:
            st.markdown("#### 📈 Avg Daily Calories vs Target")
            fig_mavg = go.Figure()
            fig_mavg.add_trace(go.Scatter(
                x=monthly_df["MonthLabel"], y=monthly_df["Avg Daily Cal"],
                mode="lines+markers+text", name="Avg Daily kcal",
                line=dict(color="#63b3ed", width=2.5),
                marker=dict(size=9, color=["#ef4444" if c > daily_target else "#48c78e" for c in monthly_df["Avg Daily Cal"]]),
                text=[f"{c:.0f}" for c in monthly_df["Avg Daily Cal"]],
                textposition="top center", textfont=dict(color="#e2e8f0"),
            ))
            fig_mavg.add_hline(y=daily_target, line_dash="dash", line_color="#f6ad55",
                               annotation_text=f"Daily Target: {daily_target}", annotation_font_color="#f6ad55")
            fig_mavg.update_layout(**CHART_LAYOUT, height=300, yaxis_title="Avg kcal/day")
            st.plotly_chart(fig_mavg, use_container_width=True)

        with mg2:
            st.markdown("#### 🥩 Monthly Macro Totals")
            fig_mm = go.Figure()
            fig_mm.add_trace(go.Bar(x=monthly_df["MonthLabel"], y=monthly_df["Protein"], name="Protein", marker_color="#9a75ea"))
            fig_mm.add_trace(go.Bar(x=monthly_df["MonthLabel"], y=monthly_df["Carbs"],   name="Carbs",   marker_color="#63b3ed"))
            fig_mm.add_trace(go.Bar(x=monthly_df["MonthLabel"], y=monthly_df["Fats"],    name="Fats",    marker_color="#f6ad55"))
            fig_mm.update_layout(**CHART_LAYOUT, barmode="group", height=300, yaxis_title="grams")
            st.plotly_chart(fig_mm, use_container_width=True)

        mg3, mg4 = st.columns(2)
        with mg3:
            st.markdown("#### ⭐ Monthly Avg Health Score")
            fig_mhs = go.Figure()
            fig_mhs.add_trace(go.Bar(
                x=monthly_df["MonthLabel"], y=monthly_df["Health_Score"].fillna(0),
                marker_color=["#48c78e" if s>=7 else "#f6ad55" if s>=5 else "#ef4444" for s in monthly_df["Health_Score"].fillna(0)],
                text=[f"{s:.1f}" for s in monthly_df["Health_Score"].fillna(0)],
                textposition="outside", textfont=dict(color="#e2e8f0"),
            ))
            fig_mhs.add_hline(y=7, line_dash="dot", line_color="#48c78e", annotation_text="Good (7)", annotation_font_color="#48c78e")
            fig_mhs.update_layout(**CHART_LAYOUT, height=280, yaxis_range=[0,10.5])
            st.plotly_chart(fig_mhs, use_container_width=True)

        with mg4:
            st.markdown("#### 🥗 Monthly Fiber Intake")
            fig_mfib = go.Figure()
            fig_mfib.add_trace(go.Bar(
                x=monthly_df["MonthLabel"], y=monthly_df["Fiber"],
                marker_color=["#48c78e" if f >= 900 else "#f6ad55" for f in monthly_df["Fiber"]],
                text=[f"{f:.0f}g" for f in monthly_df["Fiber"]],
                textposition="outside", textfont=dict(color="#e2e8f0"),
            ))
            fig_mfib.add_hline(y=900, line_dash="dash", line_color="#f6ad55", annotation_text="Monthly Rec: 900g", annotation_font_color="#f6ad55")
            fig_mfib.update_layout(**CHART_LAYOUT, height=280, yaxis_title="grams")
            st.plotly_chart(fig_mfib, use_container_width=True)

        # Monthly calorie gap
        st.markdown("#### 📊 Monthly Avg Daily Calorie Gap")
        monthly_df["Gap"] = monthly_df["Avg Daily Cal"] - daily_target
        fig_mgap = go.Figure(go.Bar(
            x=monthly_df["MonthLabel"], y=monthly_df["Gap"],
            marker_color=["#ef4444" if g > 0 else "#48c78e" for g in monthly_df["Gap"]],
            text=[f"+{g:.0f}" if g>0 else f"{g:.0f}" for g in monthly_df["Gap"]],
            textposition="outside", textfont=dict(color="#e2e8f0"),
        ))
        fig_mgap.add_hline(y=0, line_color="#f6ad55", line_width=1.5)
        fig_mgap.update_layout(**CHART_LAYOUT, height=280, yaxis_title="Avg kcal/day vs target")
        st.plotly_chart(fig_mgap, use_container_width=True)

    # ── Email Reports (outside tabs) ──
    st.markdown("---")
    st.markdown("### 📧 Email Reports")
    ec1, ec2 = st.columns(2)
    with ec1:
        if st.button("📧 Send Daily Summary Email", use_container_width=True):
            today_str = now_ist().date().isoformat()
            today_meals = list(meals_col.find({"user_id": st.session_state.user_id, "meal_date": today_str}))
            today_total = {
                "calories_kcal": sum(m.get("final_totals",{}).get("calories_kcal",0) for m in today_meals),
                "protein_g":     sum(m.get("final_totals",{}).get("protein_g",0) for m in today_meals),
                "carbs_g":       sum(m.get("final_totals",{}).get("carbs_g",0) for m in today_meals),
                "fats_g":        sum(m.get("final_totals",{}).get("fats_g",0) for m in today_meals),
            }
            with st.spinner("Sending email..."):
                ok = send_daily_summary_email(st.session_state.user_email, st.session_state.username, profile, today_meals, today_total)
            if ok:
                st.success(f"✅ Daily summary sent to {st.session_state.user_email}!")
    with ec2:
        st.info(f"📬 Reports sent to: **{st.session_state.user_email}**")

# ─────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────
def render_sidebar():
    with st.sidebar:
        st.markdown("""
        <div class="app-logo">
            <div style="font-size:2.5rem;">🥗</div>
            <h2>DATA-DRIVEN SMART FOOD ANALYTICS <br>AND NUTRITION ADVISOR</h2>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div style="background:rgba(99,179,237,0.1);border:1px solid rgba(99,179,237,0.2);
                    border-radius:10px;padding:12px;margin:12px 0;text-align:center;">
            <p style="color:#63b3ed;font-weight:600;margin:0;font-size:0.9rem;">👋 {st.session_state.username}</p>
            <p style="color:#718096;font-size:0.75rem;margin:2px 0 0;">{st.session_state.user_email}</p>
        </div>
        """, unsafe_allow_html=True)

        # Profile status
        if st.session_state.profile_complete:
            profile = profiles_col.find_one({"user_id": st.session_state.user_id}) or {}
            target = profile.get("daily_calorie_target", 2000)
            today_str = now_ist().date().isoformat()
            today_meals = list(meals_col.find({"user_id": st.session_state.user_id, "meal_date": today_str}))
            today_cal = sum(m.get("final_totals",{}).get("calories_kcal",0) for m in today_meals)
            pct = min(today_cal / target, 1.0) if target else 0

            st.markdown("**📊 Today's Progress**")
            st.progress(pct)
            color = "#ef4444" if today_cal > target else "#48c78e"
            st.markdown(f"<p style='color:{color};font-size:0.85rem;text-align:center;margin:0;'>{today_cal:.0f} / {target:.0f} kcal</p>", unsafe_allow_html=True)

            # IST time
            st.markdown(f"<p style='color:#718096;font-size:0.75rem;text-align:center;margin:8px 0 0;'>🕐 {now_ist().strftime('%I:%M %p IST')}</p>", unsafe_allow_html=True)

        st.markdown("---")

        # Navigation
        section = st.radio(
            "Navigate",
            ["🏠 Home", "👤 User Information", "🥘 AI Tracker", "📈 Analytics"],
            index=["🏠 Home","👤 User Information","🥘 AI Tracker","📈 Analytics"].index(st.session_state.current_section),
        )
        st.session_state.current_section = section

        st.markdown("---")

        # Profile badge
        if st.session_state.profile_complete:
            st.markdown('<div class="success-banner" style="font-size:0.8rem;text-align:center;">✅ Profile Complete</div>', unsafe_allow_html=True)
        else:
            st.markdown('<div class="warning-banner" style="font-size:0.8rem;text-align:center;">⚠️ Complete your profile to unlock all features</div>', unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)

        if st.button("🚪 Logout", use_container_width=True):
            for key in list(st.session_state.keys()):
                del st.session_state[key]
            init_session_state()
            st.rerun()

# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    if not st.session_state.authenticated:
        render_auth()
        return

    render_sidebar()

    section = st.session_state.current_section
    if section == "🏠 Home":
        render_home()
    elif section == "👤 User Information":
        render_user_info()
    elif section == "🥘 AI Tracker":
        render_tracker()
    elif section == "📈 Analytics":
        render_analytics()

if __name__ == "__main__":
    main()

Writing app.py


In [ ]:
from pyngrok import ngrok

ngrok_key = os.environ.get("Ngrok_API_KEY")
port = 8501

ngrok.set_auth_token(ngrok_key)
ngrok.connect(port).public_url

'https://71be-34-106-213-76.ngrok-free.app'

In [ ]:
!rm -rf logs.txt && streamlit run app.py &>/content/logs.txt